# Lab 7: Barcode and QR Code Detection

**Duration:** 2 hours 30 minutes. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Many things a camera sees carry **information in a code**: parcels, tickets, name cards, medicine boxes and products on a shelf. A QR code or a barcode is made so that a machine can read it fast and without mistakes. In this lab the Jetson **reads** codes from the camera. You build a **QR attendance checker** (a student shows a QR code and the program marks the name present) and a **barcode reader** that shows the type and the number of a product code. No AI model is needed: this is classical computer vision, and it is a good first use of the camera before the AI labs.

**By the end of this lab you can:**
1. Create a QR code and explain what is stored in it.
2. Read QR codes from the live camera and check them against a list.
3. Make and read a product barcode (EAN-13) and explain the check digit.
4. Explain why reading a code is fast and exact, and why it is not machine learning.
5. Test the reader and find its limits (distance, light, tilt).

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Privacy.** A QR code can hold a name or a number. Use only the codes you make in this lab and show them only to your classmates. Do not scan the cards of other people.

## 2. Background

**QR codes.** A QR code is a square picture that stores up to a few hundred characters of text (a name, a web address, a number). The three big squares in the corners tell the reader where the code is and how it is turned, so a QR code can be read from any angle. A QR code also has **error correction**: it can still be read when a small part is dirty or covered.

**Barcodes.** The barcode on a product stores only a number, in black and white lines. The most common one on shop products is **EAN-13**: 13 digits. The last digit is a **check digit**, calculated from the other 12. The reader uses it to find out if it read the number correctly.

| | QR code | Barcode (EAN-13) |
|---|---|---|
| Shape | Square | Lines |
| Stores | Text, up to hundreds of characters | A number of 13 digits |
| Read from | Any angle | The lines must be fairly straight |
| Error check | Error correction (can repair mistakes) | Check digit (only finds mistakes) |

**Not machine learning.** Reading a code is a fixed method. OpenCV finds the QR pattern and the library **zbar** reads barcodes. There is no model that is trained. So it is fast and exact, and it either works or it does not. Compare with the next labs: a neural network *guesses* with a confidence.

**Things that make a code hard to read.** Small code in the picture, blur, a shiny sticker or a phone screen with glare, a code that is cut at the edge, a very tilted code, or no white border around the code (the *quiet zone*).

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(7)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: Check the Reading Library
Barcodes need the library **zbar**. Setup of a new Jetson installs it. This cell checks it. If it is missing, it tells you what to do.

In [ ]:
import glob
import os
import subprocess
import sys

print("This notebook is saved in:", os.getcwd())
print("zbar library:", "found" if glob.glob("/usr/lib/*/libzbar.so*") else "MISSING")

If it says `MISSING`, install it now. The first cell asks for your password (it is not shown). The second cell installs the library (about 1 minute, it needs the internet).

In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo apt install -y libzbar0

### Step 2: Install the Python Part
`pyzbar` lets Python read barcodes with zbar. The QR pictures are made with OpenCV, or with the `qrcode` package when the OpenCV of the Jetson cannot (the lab installs it by itself).

In [ ]:
for package, module in (("pyzbar", "pyzbar"),):
    try:
        __import__(module)
        print(package, "is installed.")
    except ImportError:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(package, "is installed.")

### Step 3: Plan Your Reader
Think about where reading codes helps.

| Idea | Reads | Data in the code |
|---|---|---|
| Class attendance | QR code on a card | Student ID |
| Parcel sorting | QR code or barcode | Parcel number and place |
| Shop shelf | Barcode | Product number |
| Door pass | QR code on a phone | Pass number |
| Lab equipment | QR code on a box | Item name |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea you chose:
- What is read (QR or barcode) and where is the camera:
- What should the program do after it reads the data:

---
# Part B: QR Attendance (50 min)

### Step 1: The Settings
`ROSTER` is the list of people. The key is the text stored in the QR code, the value is the name that is shown. Add your own team members. `COOLDOWN` is the number of seconds before the same code is marked again (the log must not fill with the same person many times).

In [ ]:
# ---------------- Settings you can change ----------------
ROSTER = {"DKA4-AMINAH": "Aminah", "DKA4-AMIRUL": "Amirul", "DKA4-AMIRA": "Amira", "DKA4-AMEERA": "Ameera"}
COOLDOWN = 10            # seconds before the same code is marked again
# ----------------------------------------------------------
print("People in the list:", len(ROSTER))

### Step 2: Make the QR Codes
The cell makes one QR code picture for each person with `labview.qr_picture`. The pictures are saved as `qr_<name>.png`. Look at them in the cell output. To use them, open the file in the file browser on the left, right-click it and choose **Download**, then show it on a phone, or print it. Each QR code is only a picture of the text.

In [ ]:
import csv
import re
import time
from datetime import datetime

import labview
import cv2
import numpy as np
from IPython.display import Image, display


def make_qr(text, size=6):
    """A QR code picture (white border included) for this text."""
    return labview.qr_picture(text, size)       # OpenCV, or the qrcode package when OpenCV is old


for text, name in ROSTER.items():
    picture = make_qr(text)
    cv2.imwrite(f"qr_{name.lower()}.png", picture)
    print(f"{name}: stores the text {text}")
    display(Image(filename=f"qr_{name.lower()}.png", width=160))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many characters are stored in the QR code of your name?:
- Does the picture contain your name, or only the text that you gave?:


### Step 3: Read a Code from the Picture
Before the live video, read one saved code. `labview.read_qr` gives a list with one `(text, corners)` for each code in the picture: the text and the four corner points. It uses `pyzbar`, because the OpenCV of this Jetson can find a code but cannot read it. If there is no code, the list is empty.

In [ ]:
first = sorted(glob.glob("qr_*.png"))[0]
picture = cv2.imread(first)
found = labview.read_qr(picture)                                   # a list: one (text, corners) for each code
text, corners = found[0] if found else ("", None)
print("File:", first)
print("Text read:", repr(text))
print("Corner points:", None if corners is None else corners.tolist())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Text read from the picture:
- What are the corner points used for in the live program?:


### Step 4: The Attendance Program
The program reads every QR code in the picture and checks the text against `ROSTER`. A known code is marked **present** (green) and written to `attendance.csv` and `attendance.xlsx`. An unknown code is shown in red and not logged.

| Function | What it does |
|---|---|
| `read_codes(frame)` | Finds all QR codes in the picture. Returns a list of (text, corner points). |
| `mark(text)` | Marks a known person present. Returns `True` when a new line was logged, `False` when the person was marked less than `COOLDOWN` seconds ago. |
| `log_attendance(code, name)` | Adds a line to the log files |
| `process(frame, state)` | Reads, checks, draws. The live view calls it for every picture. |

In [ ]:
run = {"last": {}, "note": "", "read": False, "text": "", "warned": False}


def read_codes(frame):
    """All QR codes in the picture: a list of (text, corners). The text is empty if a code could not be read."""
    return labview.read_qr(frame)


def write_excel():
    """Rewrite attendance.xlsx from attendance.csv, so it opens in Excel with a header."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("attendance.xlsx needs openpyxl. attendance.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Attendance"
    with open("attendance.csv", newline="") as f:
        for row in csv.reader(f):
            sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABC", (20, 18, 14)):
        sheet.column_dimensions[column].width = width
    book.save("attendance.xlsx")


def log_attendance(code, name):
    first = not os.path.exists("attendance.csv")
    with open("attendance.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "name", "code"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), name, code])
    write_excel()


def mark(text, now=None):
    """Mark a known person present. True = a new line was logged."""
    now = time.time() if now is None else now
    if text not in ROSTER:
        return False
    if now - run["last"].get(text, -1e9) < COOLDOWN:
        return False
    run["last"][text] = now
    log_attendance(text, ROSTER[text])
    return True


def process(frame, state):
    shown = frame.copy()
    for text, corners in read_codes(frame):
        known = text in ROSTER
        colour = (0, 200, 0) if known else (0, 0, 255)
        cv2.polylines(shown, [corners.reshape(-1, 1, 2)], True, colour, 3)
        label = ROSTER[text] + " present" if known else ("unknown code" if text else "cannot read")
        x, y = corners[:, 0].min(), max(corners[:, 1].min() - 10, 20)
        cv2.putText(shown, label, (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colour, 2)
        if mark(text):
            run["note"] = f"{ROSTER[text]} marked at {datetime.now().strftime('%H:%M:%S')}"
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return shown


print("The program is ready.")

### Step 5: Run the Attendance Checker
Run the cell. Show a QR code (on a phone screen or on paper) to the camera. Hold it still, flat and about 20 to 30 cm from the lens. Watch the coloured frame and the name. Try a code that is not in the list: make one with the cell below. Press **Stop** when you are done.

In [ ]:
run["last"].clear()
run["note"] = ""
live = labview.live(process)

To test an unknown code, run this cell. It makes `qr_stranger.png` with a text that is not in the `ROSTER`. Open it from the file browser and show it to the camera.

In [ ]:
cv2.imwrite("qr_stranger.png", make_qr("DKA4-STRANGER"))
display(Image(filename="qr_stranger.png", width=160))

### Step 6: See the Results
Run this cell after you pressed Stop. To open the Excel file, use the file browser on the left, right-click `attendance.xlsx`, choose **Download** and open it on your laptop.

In [ ]:
def show_results():
    for name in ("attendance.csv", "attendance.xlsx"):
        print(f"{name:<16}", "found" if os.path.exists(name) else "not created yet (it appears at the first marked person)")
    if os.path.exists("attendance.csv"):
        print()
        with open("attendance.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- From how far away could the camera read a QR code on your phone?:
- What happened when you tilted the code or covered a corner?:
- What did the program show for the stranger code?:
- First line of the log after the header:


---
# Part C: Barcodes (35 min)

### Step 1: Make a Barcode
The cell draws an **EAN-13** barcode from 12 digits. It calculates the check digit and draws the lines. Each digit becomes 7 thin strips (black or white), and a barcode has a start, a middle and an end pattern.

In [ ]:
L = ["0001101", "0011001", "0010011", "0111101", "0100011", "0110001", "0101111", "0111011", "0110111", "0001011"]
G = ["0100111", "0110011", "0011011", "0100001", "0011101", "0111001", "0000101", "0010001", "0001001", "0010111"]
R = ["1110010", "1100110", "1101100", "1000010", "1011100", "1001110", "1010000", "1000100", "1001000", "1110100"]
PARITY = ["LLLLLL", "LLGLGG", "LLGGLG", "LLGGGL", "LGLLGG", "LGGLLG", "LGGGLL", "LGLGLG", "LGLGGL", "LGGLGL"]


def check_digit(twelve):
    """The 13th digit: digits in odd places count once, digits in even places count three times."""
    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(twelve))
    return (10 - total % 10) % 10


def make_barcode(twelve, module=3, height=140):
    """An EAN-13 barcode picture (white border included) for 12 digits."""
    digits = twelve + str(check_digit(twelve))
    bits = "101"
    for d, kind in zip(digits[1:7], PARITY[int(digits[0])]):
        bits += (L if kind == "L" else G)[int(d)]
    bits += "01010"
    for d in digits[7:]:
        bits += R[int(d)]
    bits += "101"
    strip = np.repeat(np.array([0 if b == "1" else 255 for b in bits], np.uint8), module)
    quiet = np.full(10 * module, 255, np.uint8)
    row = np.concatenate([quiet, strip, quiet])
    img = np.tile(row, (height, 1))
    img = cv2.copyMakeBorder(img, 15, 30, 0, 0, cv2.BORDER_CONSTANT, value=255)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.putText(img, digits, (10 * module, img.shape[0] - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    return img, digits


picture, number = make_barcode("590123412345")
cv2.imwrite("barcode.png", picture)
print("Number with the check digit:", number)
display(Image(filename="barcode.png"))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The check digit of your barcode:
- Change the 12 digits and run the cell again. Does the check digit change?:

### Step 2: Read the Barcode from the Picture
`pyzbar` finds every barcode and QR code in a picture. For each one it gives the **type**, the **data** and a **box**.

In [ ]:
from pyzbar.pyzbar import decode

def read_barcodes(img):
    """Every code found: a list of (type, text, (x, y, w, h))."""
    found = []
    for item in decode(img):
        r = item.rect
        found.append((item.type, item.data.decode("utf-8", "replace"), (r.left, r.top, r.width, r.height)))
    return found


for kind, text, box in read_barcodes(cv2.imread("barcode.png")):
    print("Type:", kind, "  Data:", text, "  Box:", box)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Type and data that were read:
- Is the data the same as the number printed under the barcode:

### Step 3: A Wrong Number
A barcode with one wrong strip is not read. The cell damages the picture on purpose and tries to read it again.

In [ ]:
damaged = cv2.imread("barcode.png")
damaged[:, 120:135] = 255              # a white stripe over some of the lines, from the top to the bottom
cv2.imwrite("barcode_damaged.png", damaged)
display(Image(filename="barcode_damaged.png"))
print("Read from the damaged picture:", read_barcodes(damaged))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Was the damaged barcode read:
- Why is it better that the reader gives nothing than a wrong number:

### Step 4: Run the Live Barcode Reader
Run the cell. Show a printed barcode, a barcode on a product box, or the picture `barcode.png` on a screen. The program draws a box and writes the type and the data. Press **Stop** when you are done.

In [ ]:
def process_codes(frame, state):
    shown = frame.copy()
    for kind, text, (x, y, w, h) in read_barcodes(frame):
        cv2.rectangle(shown, (x, y), (x + w, y + h), (0, 200, 0), 3)
        cv2.putText(shown, f"{kind} {text}", (x, max(y - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 0), 2)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


live = labview.live(process_codes)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- Types of code that were read (QR_CODE, EAN13, ...):
- From what distance (cm) did a barcode stop being read:

---
# Part D: Make It Your Own (20 min)
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Make QR codes with your own texts (for example a room name or a parcel number) and a `ROSTER` for a parcel station.
- Count the people marked present and show `present: 3 of 4` on the screen.
- Show the names of the people who are **not** present yet.
- Make the barcode reader keep a list of the numbers it has read (without repeats) and save it in a CSV file.
- Make a small price list (a dictionary from barcode number to product name) and show the product name on the screen.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- Did it work the first time? What went wrong:

---
# Part E: Test and Present (25 min)

### Step 1: Test
Delete the old logs, then test these situations and write what happened.

In [ ]:
for name in ("attendance.csv", "attendance.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("attendance*")))

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | A known QR code on a phone, 25 cm away | Name shown, one log line | |
| 2 | The same code again after 3 seconds | No new log line (cooldown) | |
| 3 | The same code 15 seconds later | A new log line | |
| 4 | A QR code that is not in the list | Red frame, no log line | |
| 5 | The QR code far away (60 cm) | Read or not? Write the distance at which it fails | |
| 6 | The barcode picture on a screen | Type and data shown | |
| 7 | A real product barcode, tilted | Read or not? Write the angle at which it fails | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4: were all four as expected:
- Test 5: the distance where the code was no longer read:
- Test 6 and 7: what did you see:

### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: make a QR code, mark a person present, show a rejected code, read a barcode and show the log.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea and what the program reads:
- FPS shown:
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:

> **Where next.** QR codes and barcodes are *made* to be read. In the next lab you read ordinary **printed text**, which is harder: it uses OCR.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | QR codes are made and the text stored in each one is explained | |
| 2 | The live checker marks a known person present and logs one line | |
| 3 | An unknown code is shown in red and not logged | |
| 4 | The cooldown stops the same person from being logged again at once | |
| 5 | A barcode is made, its check digit is explained and it is read from the picture | |
| 6 | The live reader shows the type and the data of a real or shown barcode | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Is reading a QR code machine learning? Explain why it is fast and exact.:
- 2. What is stored in a QR code? Does the picture contain a name by itself?:
- 3. Why does the attendance program use a cooldown?:
- 4. What is the check digit of a barcode for? Can it repair a wrong number?:
- 5. What is the difference between the error correction of a QR code and the check digit of a barcode?:
- 6. A QR code on a shiny sticker is not read. Name two things that could help.:
- 7. Why is a quiet zone (white border) needed around a code?:
- 8. Why is a QR code a bad way to store a password in a public place?:
- 9. Which is an exact reader, and which one guesses? (a) a QR reader (b) an image classifier:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Unable to find zbar` | `libzbar0` is not installed | Run the `apt install` cell of Part A |
| `No module named 'pyzbar'` | Step 2 of Part A was not run | Run Step 2 of Part A |
| `AttributeError ... QRCodeEncoder` | The OpenCV on this Jetson is old | Ask your teacher. As a workaround run `!pip install qrcode` and use `qrcode.make(text)` to make the picture. |
| The QR code is not read | Too small, blurry, tilted a lot, shiny, or cut at an edge | Move to 20 to 30 cm, hold still, turn off the shine of the phone, show the whole code with its white border |
| The name is wrong or `unknown code` | The text in the code is not in `ROSTER` | Check the spelling and capitals of the keys of `ROSTER` and make the codes again |
| The barcode is not read | Lines too small or blurred, or the picture is turned | Hold it flat at 20 to 30 cm, keep the lines horizontal in the picture, add light |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(7)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- OpenCV, QR code detection: https://docs.opencv.org/4.x/de/dc3/classcv_1_1QRCodeDetector.html
- pyzbar: https://github.com/NaturalHistoryMuseum/pyzbar
- GS1, how the EAN-13 check digit is calculated: https://www.gs1.org/services/how-calculate-check-digit-manually